# CLIP4CAD-HUS Inference & Evaluation

Evaluate trained **Hierarchical Unified Space (HUS)** models on retrieval tasks.

## Features

- Multi-level retrieval (Unified, Global, Detail)
- Gate value analysis (interpretability)
- Cross-modal retrieval metrics (R@1, R@5, R@10, mAP)
- Attention visualization
- Geometry-only inference (no text needed)

## Key Differences from GFA Evaluation

| Aspect | GFA | HUS |
|--------|-----|-----|
| Embeddings | `z_brep`, `z_pc`, `z_text` | + `z_*_global`, `z_*_detail` |
| Interpretability | Grounding matrices | Gate values (global/detail balance) |
| Geometry-only | Needs `self_ground_queries` | Direct (same pipeline) |

---
## Cell 1: Imports & Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, 'D:/Defect_Det/MMCAD/MMCAD')

import gc
import json
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from omegaconf import OmegaConf
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn
from clip4cad.models.clip4cad_hus import CLIP4CAD_HUS_v2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Cell 2: Configuration

In [ ]:
# =============================================================================
# PATHS
# =============================================================================
PATHS = {
    "data_root": "d:/Defect_Det/MMCAD/data",
    "pc_file": "c:/Users/User/Desktop/pc_embeddings_full.h5",
    "brep_file": "c:/Users/User/Desktop/brep_features.h5",
    "text_dir": "c:/Users/User/Desktop/text_splits/",
    "config_path": "d:/Defect_Det/MMCAD/MMCAD/configs/model/clip4cad_hus.yaml",
    "checkpoint": "outputs/hus/checkpoint_epoch35.pt",  # Change to your checkpoint
}

# Evaluation config
EVAL_CONFIG = {
    "batch_size": 64,
    "split": "val",  # "val" or "test"
}

print("Configuration:")
print(f"  Checkpoint: {PATHS['checkpoint']}")
print(f"  Split: {EVAL_CONFIG['split']}")
print(f"  Batch size: {EVAL_CONFIG['batch_size']}")

---
## Cell 3: Load Model and Dataset

In [ ]:
# Load config
config = OmegaConf.load(PATHS["config_path"])

# Create model
print("Creating model...")
model = CLIP4CAD_HUS_v2(config)

# Load checkpoint
checkpoint_path = Path(PATHS["checkpoint"])
if checkpoint_path.exists():
    print(f"Loading checkpoint: {checkpoint_path.name}")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"  Epoch: {ckpt.get('epoch', '?')}")
    print(f"  Stage: {ckpt.get('stage', '?')}")
else:
    print(f"WARNING: Checkpoint not found: {checkpoint_path}")
    print("Using randomly initialized model!")

model = model.to(device).eval()
param_counts = model.count_parameters()
print(f"  Parameters: {param_counts['total']:,}")

# Load dataset
print(f"\nLoading {EVAL_CONFIG['split']} dataset...")
eval_dataset = GFAMappedDataset(
    data_root=PATHS["data_root"],
    split=EVAL_CONFIG["split"],
    pc_file=PATHS["pc_file"],
    text_file=PATHS["text_dir"],
    brep_file=PATHS["brep_file"],
    num_rotations=1,
    load_to_memory=False,
    use_live_text=False,
)
print(f"  Samples: {len(eval_dataset):,}")

---
## Cell 4: Generate Embeddings

In [ ]:
# Create dataloader
loader = DataLoader(
    eval_dataset,
    batch_size=EVAL_CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=gfa_collate_fn,
)

# Storage for embeddings
embeddings = {
    # Unified (primary)
    "z_text": [], "z_brep": [], "z_pc": [],
    # Global level
    "z_text_global": [], "z_brep_global": [], "z_pc_global": [],
    # Detail level
    "z_text_detail": [], "z_brep_detail": [], "z_pc_detail": [],
    # Gate values
    "gate_text": [], "gate_brep": [], "gate_pc": [],
}
sample_ids = []

print("Generating embeddings...")
with torch.no_grad():
    for batch in tqdm(loader, desc="Embeddings"):
        sample_ids.extend(batch["sample_id"])
        outputs = model(batch)
        
        # Collect embeddings
        for key in embeddings:
            if key in outputs:
                emb = outputs[key].float().cpu()
                if not key.startswith("gate"):
                    emb = F.normalize(emb, p=2, dim=-1)
                embeddings[key].append(emb)

# Concatenate
for key in embeddings:
    if embeddings[key]:
        embeddings[key] = torch.cat(embeddings[key])

print(f"\nGenerated {len(sample_ids):,} embeddings")
print(f"  z_text shape: {embeddings['z_text'].shape}")
print(f"  z_brep shape: {embeddings['z_brep'].shape}")
print(f"  z_pc shape: {embeddings['z_pc'].shape}")

---
## Cell 5: Compute Retrieval Metrics

In [ ]:
def compute_retrieval_metrics(query_emb, gallery_emb, query_ids, gallery_ids, k_values=[1, 5, 10]):
    """
    Compute retrieval metrics.
    
    Returns:
        dict: R@k and mAP@k for each k
    """
    # Compute similarity
    sim = torch.mm(query_emb, gallery_emb.T)
    _, rankings = torch.topk(sim, k=max(k_values), dim=1)
    
    n = len(query_ids)
    results = {}
    
    for k in k_values:
        hits = 0
        ap_sum = 0.0
        
        for i in range(n):
            qid = str(query_ids[i])
            for rank, idx in enumerate(rankings[i, :k]):
                if str(gallery_ids[idx]) == qid:
                    hits += 1
                    ap_sum += 1.0 / (rank + 1)
                    break
        
        results[f"R@{k}"] = hits / n
        results[f"mAP@{k}"] = ap_sum / n
    
    return results


# Define retrieval tasks
tasks = [
    # Unified embeddings (primary)
    ("Text->BRep (Unified)", "z_text", "z_brep"),
    ("Text->PC (Unified)", "z_text", "z_pc"),
    ("PC->BRep (Unified)", "z_pc", "z_brep"),
    ("BRep->PC (Unified)", "z_brep", "z_pc"),
    # Global embeddings
    ("Text->BRep (Global)", "z_text_global", "z_brep_global"),
    ("Text->PC (Global)", "z_text_global", "z_pc_global"),
    # Detail embeddings
    ("Text->BRep (Detail)", "z_text_detail", "z_brep_detail"),
    ("Text->PC (Detail)", "z_text_detail", "z_pc_detail"),
]

# Compute metrics
all_results = {}
print("Computing retrieval metrics...\n")

for task_name, query_key, gallery_key in tasks:
    query_emb = embeddings[query_key]
    gallery_emb = embeddings[gallery_key]
    
    results = compute_retrieval_metrics(
        query_emb, gallery_emb, sample_ids, sample_ids
    )
    all_results[task_name] = results

# Print results
print("=" * 80)
print("RETRIEVAL METRICS")
print("=" * 80)
print(f"{'Task':<30} {'R@1':>10} {'R@5':>10} {'R@10':>10} {'mAP@1':>10}")
print("-" * 80)

for task_name, results in all_results.items():
    print(f"{task_name:<30} "
          f"{results['R@1']:>10.4f} {results['R@5']:>10.4f} {results['R@10']:>10.4f} "
          f"{results['mAP@1']:>10.4f}")

print("=" * 80)

---
## Cell 6: Gate Value Analysis

Gate values show the learned balance between global and detail features:
- Gate = 0.5: Equal contribution from global and detail
- Gate > 0.5: More global (coarse shape info)
- Gate < 0.5: More detail (fine-grained features)

In [ ]:
# Gate statistics
print("Gate Value Statistics")
print("=" * 60)
print(f"{'Modality':<15} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 60)

for modality in ["brep", "pc", "text"]:
    gate = embeddings[f"gate_{modality}"]
    print(f"{modality.upper():<15} {gate.mean():>10.4f} {gate.std():>10.4f} "
          f"{gate.min():>10.4f} {gate.max():>10.4f}")

print("=" * 60)
print("Note: Gate > 0.5 means more global, < 0.5 means more detail")

# Plot gate distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, modality in enumerate(["brep", "pc", "text"]):
    ax = axes[idx]
    gate = embeddings[f"gate_{modality}"].numpy()
    
    ax.hist(gate, bins=50, alpha=0.7, color=['blue', 'green', 'orange'][idx])
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='Balanced (0.5)')
    ax.axvline(x=gate.mean(), color='black', linestyle='-', alpha=0.7, label=f'Mean ({gate.mean():.3f})')
    ax.set_xlabel('Gate Value')
    ax.set_ylabel('Count')
    ax.set_title(f'{modality.upper()} Gate Distribution')
    ax.legend()
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

---
## Cell 7: Embedding Space Visualization

In [ ]:
from sklearn.manifold import TSNE

# Sample embeddings for visualization
n_samples = min(1000, len(sample_ids))
indices = np.random.choice(len(sample_ids), n_samples, replace=False)

# Get unified embeddings
z_text_sample = embeddings["z_text"][indices].numpy()
z_brep_sample = embeddings["z_brep"][indices].numpy()
z_pc_sample = embeddings["z_pc"][indices].numpy()

# Combine for t-SNE
combined = np.concatenate([z_text_sample, z_brep_sample, z_pc_sample], axis=0)
labels = ["Text"] * n_samples + ["BRep"] * n_samples + ["PC"] * n_samples

print(f"Running t-SNE on {len(combined)} embeddings...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(combined)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

colors = {"Text": "orange", "BRep": "blue", "PC": "green"}
for modality in ["Text", "BRep", "PC"]:
    mask = [l == modality for l in labels]
    ax.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=colors[modality],
        label=modality,
        alpha=0.5,
        s=10
    )

ax.set_title("Unified Embedding Space (t-SNE)")
ax.legend()
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

print("Well-aligned modalities should overlap in the embedding space.")

---
## Cell 8: Geometry-Only Inference

HUS can encode geometry (B-Rep, PC) without text input.
This is useful for retrieval tasks when only geometry is available.

In [ ]:
print("Testing geometry-only inference...")

# Get a batch
test_loader = DataLoader(
    eval_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    collate_fn=gfa_collate_fn,
)
batch = next(iter(test_loader))

# Full forward (with text)
with torch.no_grad():
    outputs_full = model(batch)

# Geometry-only forward
with torch.no_grad():
    outputs_geo = model.encode_geometry_inference(batch)

print("\nFull forward outputs:")
for key in sorted(outputs_full.keys()):
    if isinstance(outputs_full[key], torch.Tensor):
        print(f"  {key}: {outputs_full[key].shape}")

print("\nGeometry-only outputs:")
for key in sorted(outputs_geo.keys()):
    if isinstance(outputs_geo[key], torch.Tensor):
        print(f"  {key}: {outputs_geo[key].shape}")

# Compare B-Rep embeddings (should be identical)
z_brep_full = F.normalize(outputs_full["z_brep"], p=2, dim=-1)
z_brep_geo = F.normalize(outputs_geo["z_brep"], p=2, dim=-1)
similarity = (z_brep_full * z_brep_geo).sum(dim=-1).mean()

print(f"\nB-Rep embedding similarity: {similarity:.6f}")
print("(Should be 1.0 - geometry encoding is independent of text)")

---
## Cell 9: Qualitative Examples

Show top-K retrieval results for random samples.

In [ ]:
def show_retrieval_examples(query_key, gallery_key, n_queries=5, k=5):
    """
    Show retrieval examples.
    """
    query_emb = embeddings[query_key]
    gallery_emb = embeddings[gallery_key]
    
    # Compute similarities
    sim = torch.mm(query_emb, gallery_emb.T)
    _, rankings = torch.topk(sim, k=k, dim=1)
    
    # Random queries
    query_indices = np.random.choice(len(sample_ids), n_queries, replace=False)
    
    print(f"\n{query_key} -> {gallery_key} Retrieval Examples")
    print("=" * 80)
    
    for qi in query_indices:
        query_id = sample_ids[qi]
        retrieved_ids = [sample_ids[rankings[qi, j]] for j in range(k)]
        
        # Check which are correct
        matches = ["O" if str(rid) == str(query_id) else "X" for rid in retrieved_ids]
        
        print(f"\nQuery: {query_id}")
        print(f"  Top-{k}: {' | '.join(str(rid) for rid in retrieved_ids)}")
        print(f"  Match:  {' | '.join(f' {m} ' for m in matches)}")


# Show examples
show_retrieval_examples("z_text", "z_brep", n_queries=5, k=5)

---
## Cell 10: Save Embeddings

Save embeddings for downstream tasks.

In [ ]:
import h5py

output_dir = Path(PATHS["checkpoint"]).parent
output_file = output_dir / f"embeddings_{EVAL_CONFIG['split']}.h5"

print(f"Saving embeddings to: {output_file}")

with h5py.File(output_file, "w") as f:
    # Save sample IDs
    f.create_dataset("sample_ids", data=np.array([str(s) for s in sample_ids], dtype="S"))
    
    # Save embeddings
    for key, emb in embeddings.items():
        if isinstance(emb, torch.Tensor):
            f.create_dataset(key, data=emb.numpy())
            print(f"  {key}: {emb.shape}")

print(f"\nDone! File size: {output_file.stat().st_size / 1e6:.1f} MB")

---
## Cell 11: Compare with GFA (Optional)

If you have GFA embeddings, compare retrieval performance.

In [ ]:
# Uncomment and modify paths to compare with GFA

# gfa_embeddings_path = "outputs/ablations/asymmetric_grounding/embeddings_val.h5"

# if Path(gfa_embeddings_path).exists():
#     with h5py.File(gfa_embeddings_path, "r") as f:
#         gfa_z_text = torch.from_numpy(f["z_text"][:])
#         gfa_z_brep = torch.from_numpy(f["z_brep"][:])
#         gfa_ids = [s.decode() for s in f["sample_ids"][:]]
#     
#     # Compute GFA metrics
#     gfa_results = compute_retrieval_metrics(gfa_z_text, gfa_z_brep, gfa_ids, gfa_ids)
#     hus_results = all_results["Text->BRep (Unified)"]
#     
#     print("\nComparison: GFA vs HUS")
#     print("=" * 60)
#     print(f"{'Metric':<15} {'GFA':>15} {'HUS':>15} {'Delta':>15}")
#     print("-" * 60)
#     for metric in ["R@1", "R@5", "R@10", "mAP@1"]:
#         gfa_val = gfa_results[metric]
#         hus_val = hus_results[metric]
#         delta = hus_val - gfa_val
#         sign = "+" if delta > 0 else ""
#         print(f"{metric:<15} {gfa_val:>15.4f} {hus_val:>15.4f} {sign}{delta:>14.4f}")
#     print("=" * 60)
# else:
#     print(f"GFA embeddings not found at: {gfa_embeddings_path}")

print("Uncomment the code above to compare with GFA embeddings.")

---
## Memory Cleanup

In [ ]:
import psutil

gc.collect()
torch.cuda.empty_cache()

process = psutil.Process()
ram_gb = process.memory_info().rss / 1e9
print(f"Process RAM: {ram_gb:.1f} GB")

if torch.cuda.is_available():
    gpu_alloc = torch.cuda.memory_allocated() / 1e9
    print(f"GPU Allocated: {gpu_alloc:.1f} GB")